##### Copyright 2024 Google LLC。

In [ ]:
# @title Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# PaliGemma2 - 使用Transformers.js 執行

作者：西塔姆‧梅爾
*   GitHub：[github.com/sitemgithub-MSIT](https://github.com/sitamgithub-MSIT/)
*   X：[@sitammeur](https://x.com/sitammeur)

描述：此 notebook 示範如何使用 Node.js 和 [Transformers.js](https://huggingface.co/docs/transformers.js/index) 在 PaliGemma2 模型上執行 inference。 Transformers.js 讓您直接在瀏覽器中執行Hugging Face 的變壓器模型，提供與Python 類似的JavaScript API。 它使用 ONNX Runtime 支援 NLP、電腦視覺、音訊和多模式任務，並允許輕鬆轉換 PyTorch、TensorFlow 和 JAX 模型。
<table align="left"> <td>    <a target="_blank" href="https://colab.research.google.com/github/google-gemma/cookbook/blob/main/.archive/PaliGemma/[PaliGemma_2]Using_with_Transformersjs.ipynb"><img src="https://www.tensorflow.org/images/colab_logo_32px.png" />Run in Google Colab</a>
</td>
</table>

## 設定

### 選擇 Colab runtime
要完成本教學，您需要擁有 Colab runtime 以及足夠的資源來執行 PaliGemma 2 模型。在這種情況下，您可以使用 CPUruntime：
1. 在 Colab 視窗的右上角，選擇 **▾（其他連接選項）**。
2. 選擇**更改 runtime 類型**。
3. 在 **硬體加速器** 下，選擇 **CPU**。

## 安裝

讓我們開始安裝依賴項。

In [ ]:
# Install Node.js
!curl -fsSL https://deb.nodesource.com/setup_20.x | sudo -E bash -
!sudo apt-get install -y nodejs

## 建立 Node.js 項目

建立一個新的 Node.js 專案並透過 [NPM](https://www.npmjs.com/package/@huggingface/transformers) 安裝所需的轉換器套件。

In [ ]:
# Create project directory
!mkdir paligemma2-node
%cd paligemma2-node

# Initialize NPM project
!npm init -y
!npm i @huggingface/transformers

In [ ]:
%%writefile package.json

{
  "name": "paligemma2-node",
  "version": "1.0.0",
  "main": "index.js",
  "type": "module",
  "scripts": {
    "test": "echo \"Error: no test specified\" && exit 1"
  },
  "keywords": [],
  "author": "",
  "license": "ISC",
  "description": "",
  "dependencies": {
    "@huggingface/transformers": "^3.1.2"
  }
}

## Transformers.js 推論

現在，讓我們使用Transformers.js 在PaliGemma2 模型上執行inference。首先，載入模型和處理器，然後準備輸入（文字查詢+圖像）以執行inference並獲得所需圖像標題的輸出。作為參考，您可以在 ONNX 模型部分下的 Hugging Face 模型中心查看模型頁面[此處](https://huggingface.co/onnx-community/paligemma2-3b-pt-224)。

In [ ]:
# Show the image from the URL
from PIL import Image
import requests

url = "https://jethac.github.io/assets/juice.jpg"
img = Image.open(requests.get(url, stream=True).raw) 
img

這是一張貓坐在袋子上的圖像，現在讓我們看看模型預測什麼。

In [ ]:
%%writefile index.js

// Import the required modules
import {
  AutoProcessor,
  PaliGemmaForConditionalGeneration,
  load_image,
} from "@huggingface/transformers";

// Load processor and model
const model_id = "onnx-community/paligemma2-3b-pt-224"; // Change this to use a different PaliGemma model
const processor = await AutoProcessor.from_pretrained(model_id);
const model = await PaliGemmaForConditionalGeneration.from_pretrained(
  model_id,
  {
    dtype: {
      embed_tokens: "q8", // or 'fp16'
      vision_encoder: "q8", // or 'q4', 'fp16'
      decoder_model_merged: "q4", // or 'q4f16'
    },
  }
);
console.log("Model and processor loaded successfully!");

// Prepare inputs
const url = "https://jethac.github.io/assets/juice.jpg";
const raw_image = await load_image(url);
const prompt = "<image>"; // Caption, by default
const inputs = await processor(raw_image, prompt);
console.log("Inputs prepared successfully!");

try {
  // Generate a response
  const response = await model.generate({
    ...inputs,
    max_new_tokens: 100, // Maximum number of tokens to generate
  });

  // Extract generated IDs from the response
  const generatedIds = response.slice(null, [inputs.input_ids.dims[1], null]);

  // Decode the generated IDs to get the answer
  const decodedAnswer = processor.batch_decode(generatedIds, {
    skip_special_tokens: true,
  });

  // Log the generated caption
  console.log("Generated caption:", decodedAnswer[0]);
} catch (error) {
  console.error("Error generating response:", error);
}

## 執行應用程式

In [ ]:
# Run the node.js application
!node index.js

## 結論

恭喜！您已透過 Node.js 環境使用 Transformers.js 在 PaliGemma2 模型上成功執行 inference。現在您可以將其整合到您的專案中。